In [1]:
!pip install transformers

In [2]:
import torch
import torch.nn.functional as funcy
from transformers import AutoModelForCausalLM,AutoTokenizer

In [3]:
dev='cuda' if torch.cuda.is_available() else 'cpu'

In [4]:
tar='gpt2'
draft='distilgpt2'

In [5]:
mtar=AutoModelForCausalLM.from_pretrained(tar).to(dev).eval()

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [6]:
mdraft=AutoModelForCausalLM.from_pretrained(draft).to(dev).eval()

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [7]:
tok=AutoTokenizer.from_pretrained(tar)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [8]:
if tok.pad_token is None:
    tok.pad_token=tok.eos_token

In [12]:
class SpeculativeDecoder:
    def __init__(self,mtar,mdraft,tok,k=4):
        self.tar=mtar
        self.draft=mdraft
        self.tok=tok
        self.k=k
    @torch.no_grad()
    def gen(self,prompt,mnewt=50,temp=1.0):
        inp=self.tok(prompt,return_tensors='pt').input_ids.to(dev)
        generated=inp.clone()
        ptar=None
        pdraft=None
        for s in range(mnewt):
            tdraft,prdraft=[],[]
            curr=generated[:,-1:]
            pastl=pdraft
            for _ in range(self.k):
                odraft=self.draft(curr,past_key_values=pastl,
                                 use_cache=True)
                logits=odraft.logits[:,-1,:]/temp
                probs=funcy.softmax(logits,dim=-1)
                ntok=torch.multinomial(probs,1)
                tdraft.append(ntok)
                prdraft.append(probs)
                pastl=odraft.past_key_values
                curr=ntok
            tdraft=torch.cat(tdraft,dim=1)
            tarin=torch.cat([generated,tdraft],dim=1)
            otarget=self.tar(tarin,use_cache=True)
            logt=otarget.logits[:,-self.k:,:]/temp
            prtar=funcy.softmax(logt,dim=-1)
            aall=True
            for i in range(self.k):
                token=tdraft[:,i].unsqueeze(1)
                p=prtar[:,i,:]
                q=prdraft[i]
                ptoken=p.gather(1,token)
                qtoken=p.gather(1,token)
                alpha=torch.minimum(torch.ones_like(ptoken),ptoken/qtoken)
                randv=torch.rand(1).item()
                if randv<alpha.item():
                    generated=torch.cat([generated,token],dim=1)
                else:
                    cor=torch.clamp(p-q,min=0)
                    cor=cor/cor.sum(dim=-1,keepdim=True)
                    newt=torch.multinomial(cor,1)
                    generated=torch.cat([generated,newt],dim=1)
                    aall=False
                    break
            if not aall:
                continue
        return self.tok.decode(generated[0],skip_special_tokens=True)

In [13]:
decoder=SpeculativeDecoder(mtar,mdraft,tok,k=4)

In [14]:
out=decoder.gen('The future of cars is',mnewt=50,temp=1.0)

In [15]:
print(out)

The future of cars isId. protocol attribute14.happy-12-17,B.

How do you get66 1500 Intelligence call The development of SoloIs that a new as reported:
488.6000
Your Email Address
Difficulty Complete
Now reinvention projectsgamepack Injusticeills.html; It is more Executivearium's OX play online sports videoObjectiva Norris I up with Youtube.":1,"globalTA-802-rare reduc A list of names of Riwa Pri Is Showing
and now finds themselvesbe", which givesHiddenLinks is aBevin DykesAgent 111 Billy fromtopia.
RE 13.6%Logitech Absolute Zero John Cena will hardlySparklemaniaA rush rule for A Javier PastoreAKRON, Ohio. There needs to
Judy McCCanadian Press

Dear Cheryl!
13. Chicago 500Hugh FaulkDisc press release, He can kill youLinks:ShareOpinions and of the nutrients of
